# 15 — Checkpoints, reproducibility, and inference

**Level:** Advanced · **Study time:** 60–90 minutes (including experiments)

**Prerequisites:** Notebooks 06–08.

## What you will be able to do
- Save weights and preprocessing together.
- Resume optimization with its internal state.
- Verify that a loaded model reproduces predictions.

## How to work through this lab
Run cells in order with **Shift+Enter**. Before each result, predict its shape or behavior.
Change one thing, rerun, and explain what changed. Use **Restart Kernel and Run All**
when you want a clean experiment. Lessons use local data and do not depend on earlier kernel state.
Record your observations in the experiment journal near the end.


[Course guide](../README.md) · [Previous: 14](14_autoencoders.ipynb) · [Next: 16](16_custom_autograd_and_debugging.ipynb)


## Step 1 — Prepare a reproducible experiment

A seed makes comparisons easier. Exact numerical results can still differ across devices and PyTorch versions.


In [ ]:
import random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)  # Small examples often run faster with fewer CPU threads.
device = torch.device("cpu")  # Learn on CPU first; the performance lab covers CUDA.
plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True})
print("PyTorch:", torch.__version__, "| Device:", device)


## Step 2 — Train a model with learned preprocessing

An inference pipeline includes preprocessing as well as weights. We create fresh local data so this notebook does not require a previous notebook's files. The checkpoint directory is inside the course workspace.


In [ ]:
raw_X = torch.randn(160, 3) * 4 + 8
y = raw_X @ torch.tensor([[0.5], [-1.], [2.]])
mean, std = raw_X.mean(0), raw_X.std(0)
X = (raw_X - mean) / std
model = nn.Sequential(nn.Linear(3, 16), nn.ReLU(), nn.Linear(16, 1))
optimizer = torch.optim.Adam(model.parameters(), lr=0.03)
def train_steps(model, optimizer, count):
    model.train()
    for _ in range(count):
        optimizer.zero_grad(set_to_none=True)
        loss = nn.functional.mse_loss(model(X), y)
        loss.backward(); optimizer.step()
train_steps(model, optimizer, 80)
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'notebooks').is_dir() and (p / 'requirements.txt').exists())
artifact_dir = root / 'artifacts' / '15_checkpoints'
artifact_dir.mkdir(parents=True, exist_ok=True)


## Step 3 — Save a structured checkpoint

A state dict stores parameter and buffer tensors, not the Python class implementation. Record the architecture settings and preprocessing. For resumable Adam training, save optimizer state too. This checkpoint stores simple types and tensors so it can be loaded using `weights_only=True`.


In [ ]:
checkpoint_path = artifact_dir / 'regressor.pt'
torch.save({
    'format_version': 1,
    'model': model.state_dict(),
    'optimizer': optimizer.state_dict(),
    'completed_steps': 80,
    'config': {'input_dim': 3, 'hidden_dim': 16},
    'mean': mean, 'std': std,
    'torch_rng_state': torch.get_rng_state(),
}, checkpoint_path)
print("Saved:", checkpoint_path)


## Step 4 — Load and verify inference parity

Recreate the same architecture, load on CPU, and put it in eval mode. `inference_mode()` avoids autograd bookkeeping. Load only artifacts you trust; `weights_only=True` narrows deserialization but is not a reason to trust an arbitrary file.


In [ ]:
checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
config = checkpoint['config']
restored = nn.Sequential(nn.Linear(config['input_dim'], config['hidden_dim']), nn.ReLU(),
                         nn.Linear(config['hidden_dim'], 1))
restored.load_state_dict(checkpoint['model'])
model.eval(); restored.eval()
new_raw = torch.tensor([[7., 8., 9.], [4., 5., 6.]])
def predict(raw_features):
    assert raw_features.ndim == 2 and raw_features.shape[1] == config['input_dim']
    normalized = (raw_features - checkpoint['mean']) / checkpoint['std'].clamp_min(1e-6)
    with torch.inference_mode():
        return restored(normalized)
with torch.inference_mode():
    expected = model((new_raw - mean) / std)
assert torch.allclose(predict(new_raw), expected)
print("Reloaded predictions:", predict(new_raw).flatten().tolist())


## Step 5 — Resume and compare with uninterrupted training

This full-batch model has no stochastic layers, so its continuation can be checked tightly. Real runs may also need Python/NumPy/CUDA RNG states, scheduler and scaler states, sampler position, data-loader generator state, and deterministic kernels. Saving one CPU RNG state alone does not guarantee exact resumption of every workload.


In [ ]:
resumed_optimizer = torch.optim.Adam(restored.parameters(), lr=0.03)
resumed_optimizer.load_state_dict(checkpoint['optimizer'])
train_steps(model, optimizer, 10)
torch.set_rng_state(checkpoint['torch_rng_state'])
train_steps(restored, resumed_optimizer, 10)
for expected_parameter, resumed_parameter in zip(model.parameters(), restored.parameters()):
    assert torch.allclose(expected_parameter, resumed_parameter, atol=1e-6)
print("Resumed parameters match the uninterrupted run.")


## Your turn — practice before opening the solutions

1. Save a smaller inference-only artifact with model state, configuration, mean, and std.
2. Load it and check its keys.
3. Count how many optimizer state entries were necessary to resume Adam. Explain why weights alone are insufficient.

Use the empty cell below for your attempt. The worked examples above run as supplied;
you do not need to fill in exercises to use **Run All**. Solutions are below the journal
and their code is initially collapsed in JupyterLab.


In [ ]:
# Your experiment goes here. Add more cells if you need them.


## Experiment journal

| Question | Your notes |
|---|---|
| What did I predict? | |
| What single setting did I change? | |
| What result did I measure? | |
| Why might it have changed? | |
| What would I try next? | |

## Reference solutions
Try the exercises first. Expand the following cell to compare your reasoning.


In [ ]:
inference_path = artifact_dir / 'inference_only.pt'
torch.save({'model': restored.state_dict(), 'config': config, 'mean': mean, 'std': std}, inference_path)
inference_payload = torch.load(inference_path, map_location='cpu', weights_only=True)
assert set(inference_payload) == {'model', 'config', 'mean', 'std'}
print("Adam parameter-state entries:", len(checkpoint['optimizer']['state']))
# Adam keeps per-parameter step counts and moving averages; resetting them changes future updates.


## Before moving on

A reliable save/load workflow is a tested contract: same preprocessing, compatible architecture, same outputs. For deployment, also define input validation, batching, monitoring, and model versioning.

**Explain it aloud:** What went into the model or operation, what came out, and how did you verify it?

## Official references and further reading
- [Saving and loading models](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html)
- [Reproducibility](https://docs.pytorch.org/docs/stable/notes/randomness.html)


[Course guide](../README.md) · [Previous: 14](14_autoencoders.ipynb) · [Next: 16](16_custom_autograd_and_debugging.ipynb)
